In [1]:
import os
from pathlib import Path

out_dir = Path(os.environ.get("OGS_TESTRUNNER_OUT_DIR", "_out"))

In [2]:
os.chdir(out_dir)

In [3]:
import xml.etree.ElementTree as ET  # it should be python built-in
import Mesh_settings

# Get the necessary variables
ey =  Mesh_settings.ey
duration = Mesh_settings.duration
delta_time = Mesh_settings.delta_time
number_of_timesteps = int(duration / delta_time)
GWT = - Mesh_settings.initial_GWT_depth

# Here we modify our geometry
if ey ==1: # 2D case
    tree = ET.parse('rectangle.gml')
    root = tree.getroot()

    points = root.findall('./points/point')
    for p in points:
        if p.get('id') =='1' or p.get('id') =='2':
            p.set('x', str(Mesh_settings.lx))
        if p.get('id')== '2' or p.get('id') =='3':
            p.set('y', str(-Mesh_settings.lz))
            
    # Write the modified XML back to the file
    tree.write('rectangle.gml')

else: # 3D case
    tree = ET.parse('cuboid.gml')
    root = tree.getroot()

    points = root.findall('./points/point')
    for p in points:
        if p.get('id') in {'4', '5', '6', '7'}:
            p.set('x', str(Mesh_settings.lx))
        if p.get('id') in {'2', '3', '6', '7'}:
            p.set('y', str(Mesh_settings.ly))    
        if p.get('id') in {'0', '3', '4', '7'}:
            p.set('z', str(-Mesh_settings.lz))
            
    # Write the modified XML back to the file
    tree.write('cuboid.gml')

In [4]:
import ogstools as ot

# Initiate an OGS-object

if ey==1: #2D case
    model = ot.Project(input_file="input_2D.prj", output_file="input_2D.prj")

else: #3D case
    model = ot.Project(input_file="input_3D.prj", output_file="input_3D.prj")

# Modify the ground water table
model.replace_text('if (y >='+ str (GWT) +', 0, 1000*(-y'+ str (GWT) +')*9.81)', xpath="./parameters/parameter/expression", occurrence=0)

# Modify the timesteps
model.replace_text(str(number_of_timesteps), xpath='./time_loop/processes/process/time_stepping/t_end') 
model.replace_text(str(number_of_timesteps), xpath='./time_loop/processes/process/time_stepping/timesteps/pair/repeat')
model.write_input()

# Run OGS
model.run_model(logfile="out.txt")

OGS finished with project file input_2D.prj.
Execution took 5.955530405044556 s
Project file written to output.


In [5]:
os.chdir('..')